## Model Training

In [1]:
import json
import pandas as pd
import numpy as np

In [2]:
with open("../dataset/processed/historical_processed_corrected.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data["records"])

df.head()

,source_file,academic_year,semester,discipline,section,batch,day,subject_id,subject_name,teacher_id,...,start_time,end_time,duration_hours,record_type,duplicate_record,missing_subject,missing_teacher,invalid_time_range,invalid_duration,overlaps_with_another_class
0,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,ES-CS201,None,None,...,09:45,10:45,1.0,theory,False,False,False,False,False,False
1,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,BS-CH201,None,None,...,10:45,11:45,1.0,theory,False,False,False,False,False,False
2,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,HM-HU201,None,None,...,12:30,13:30,1.0,theory,False,False,False,False,False,False
3,routine_2025_sem2.json,2025,2,CSE,CSE1,X,Tuesday,BS-CH291,None,None,...,13:30,15:30,2.0,lab,False,False,False,False,False,False
4,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Friday,ES-CS201,None,None,...,09:45,10:45,1.0,theory,False,False,False,False,False,False


In [3]:
df.shape

(48, 22)

In [4]:
df.columns

Index(['source_file', 'academic_year', 'semester', 'discipline', 'section',
       'batch', 'day', 'subject_id', 'subject_name', 'teacher_id',
       'teacher_code', 'room', 'start_time', 'end_time', 'duration_hours',
       'record_type', 'duplicate_record', 'missing_subject', 'missing_teacher',
       'invalid_time_range', 'invalid_duration',
       'overlaps_with_another_class'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   source_file                  48 non-null     str    
 1   academic_year                48 non-null     int64  
 2   semester                     48 non-null     int64  
 3   discipline                   48 non-null     str    
 4   section                      48 non-null     str    
 5   batch                        13 non-null     str    
 6   day                          48 non-null     str    
 7   subject_id                   48 non-null     str    
 8   subject_name                 0 non-null      object 
 9   teacher_id                   0 non-null      object 
 10  teacher_code                 48 non-null     str    
 11  room                         5 non-null      str    
 12  start_time                   48 non-null     str    
 13  end_time                     48 n

In [6]:
df["record_type"].value_counts()

record_type
theory      31
lab         15
remedial     2
Name: count, dtype: int64

In [7]:
df["discipline"].value_counts()

discipline
CSE     15
ECE     15
CSBS     5
EEE      5
EE       5
ME       3
Name: count, dtype: int64

In [8]:
df["day"].value_counts()

day
Tuesday      12
Wednesday    10
Thursday     10
Friday        9
Saturday      7
Name: count, dtype: int64

In [9]:
df["duration_hours"].value_counts()

duration_hours
1.0    33
2.0    15
Name: count, dtype: int64

In [10]:
df.isnull().sum()

source_file                     0
academic_year                   0
semester                        0
discipline                      0
section                         0
batch                          35
day                             0
subject_id                      0
subject_name                   48
teacher_id                     48
teacher_code                    0
room                           43
start_time                      0
end_time                        0
duration_hours                  0
record_type                     0
duplicate_record                0
missing_subject                 0
missing_teacher                 0
invalid_time_range              0
invalid_duration                0
overlaps_with_another_class     0
dtype: int64

In [11]:
df['start_time'].value_counts()

start_time
09:45    28
10:45     8
13:30     7
12:30     5
Name: count, dtype: int64

In [12]:
features_df = df.copy()
features_df.head()

,source_file,academic_year,semester,discipline,section,batch,day,subject_id,subject_name,teacher_id,...,start_time,end_time,duration_hours,record_type,duplicate_record,missing_subject,missing_teacher,invalid_time_range,invalid_duration,overlaps_with_another_class
0,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,ES-CS201,None,None,...,09:45,10:45,1.0,theory,False,False,False,False,False,False
1,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,BS-CH201,None,None,...,10:45,11:45,1.0,theory,False,False,False,False,False,False
2,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Tuesday,HM-HU201,None,None,...,12:30,13:30,1.0,theory,False,False,False,False,False,False
3,routine_2025_sem2.json,2025,2,CSE,CSE1,X,Tuesday,BS-CH291,None,None,...,13:30,15:30,2.0,lab,False,False,False,False,False,False
4,routine_2025_sem2.json,2025,2,CSE,CSE1,NaN,Friday,ES-CS201,None,None,...,09:45,10:45,1.0,theory,False,False,False,False,False,False


## Model Training

In [13]:
import json
import pandas as pd
import numpy as np



In [14]:

# ============================================================
# 1. CONSTANTS
# ============================================================

DAY_MAP = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5
}

# Time slots available in your historical timetable
POSSIBLE_TIMES = [
    585,   # 09:45
    645,   # 10:45
    750,   # 12:30
    810,   # 13:30
    870    # 14:30
]

POSSIBLE_DAYS = list(range(6))



In [15]:

# ============================================================
# 2. TIME CONVERSION
# ============================================================

def time_to_minutes(time_str):
    """
    Converts HH:MM into minutes from midnight.

    Example:
    09:45 -> 585
    13:30 -> 810
    """

    h, m = map(int, time_str.split(":"))

    return h * 60 + m



In [16]:

# ============================================================
# 3. PARSE HISTORICAL TIMETABLE JSON
# ============================================================

def parse_routine_json(json_path):

    # --------------------------------------------------------
    # Load JSON
    # --------------------------------------------------------

    with open(json_path, "r") as f:
        data = json.load(f)

    records = data.get("records", [])

    # Convert records into DataFrame
    df = pd.DataFrame(records)

    print("Original records:", len(df))

    # --------------------------------------------------------
    # Remove invalid / overlapping records
    # --------------------------------------------------------

    df = df[
        ~df["overlaps_with_another_class"]
    ].copy()

    print(
        "Records after removing overlaps:",
        len(df)
    )

    # --------------------------------------------------------
    # Convert start/end time to minutes
    # --------------------------------------------------------

    df["start_minutes"] = (
        df["start_time"].apply(time_to_minutes)
    )

    df["end_minutes"] = (
        df["end_time"].apply(time_to_minutes)
    )

    # --------------------------------------------------------
    # Convert day to numerical value
    # --------------------------------------------------------

    df["day_num"] = (
        df["day"].map(DAY_MAP)
    )

    # --------------------------------------------------------
    # Subject type
    # --------------------------------------------------------

    df["is_lab"] = (
        df["record_type"] == "lab"
    ).astype(int)

    df["is_theory"] = (
        df["record_type"] == "theory"
    ).astype(int)

    df["is_remedial"] = (
        df["record_type"] == "remedial"
    ).astype(int)

    # --------------------------------------------------------
    # Batch information
    # --------------------------------------------------------

    df["has_batch"] = (
        df["batch"].notna()
    ).astype(int)

    # --------------------------------------------------------
    # Morning / afternoon
    # --------------------------------------------------------

    df["is_morning"] = (
        df["start_minutes"] < 12 * 60
    ).astype(int)

    # --------------------------------------------------------
    # Create class identifier
    #
    # CSE + CSE1 -> CSE_CSE1
    # --------------------------------------------------------

    df["class_id"] = (
        df["discipline"]
        + "_"
        + df["section"]
    )

    # --------------------------------------------------------
    # Sort by teacher/day/time
    # --------------------------------------------------------

    df = df.sort_values(
        [
            "teacher_code",
            "day_num",
            "start_minutes"
        ]
    ).copy()

    # --------------------------------------------------------
    # Teacher daily load BEFORE current class
    #
    # Example:
    #
    # First class of teacher that day -> 0
    # Second class                  -> 1
    # Third class                   -> 2
    # --------------------------------------------------------

    df["teacher_daily_load"] = (
        df.groupby(
            [
                "teacher_code",
                "day_num"
            ]
        ).cumcount()
    )

    # --------------------------------------------------------
    # Teacher weekly workload
    # --------------------------------------------------------

    df["teacher_weekly_load"] = (
        df.groupby("teacher_code")[
            "duration_hours"
        ].transform("sum")
    )

    # --------------------------------------------------------
    # Sort by class/day/time
    # --------------------------------------------------------

    df = df.sort_values(
        [
            "class_id",
            "day_num",
            "start_minutes"
        ]
    ).copy()

    # --------------------------------------------------------
    # Section daily load BEFORE current class
    # --------------------------------------------------------

    df["section_daily_load"] = (
        df.groupby(
            [
                "class_id",
                "day_num"
            ]
        ).cumcount()
    )

    # --------------------------------------------------------
    # Section weekly workload
    # --------------------------------------------------------

    df["section_weekly_load"] = (
        df.groupby("class_id")[
            "duration_hours"
        ].transform("sum")
    )

    # --------------------------------------------------------
    # Every historical timetable entry is a positive example
    #
    # We will later compare it against candidate slots.
    # --------------------------------------------------------

    df["is_preferred"] = 1

    # Reset index
    df = df.reset_index(drop=True)

    return df


In [17]:


# ============================================================
# 4. CHECK WHETHER TWO TIME INTERVALS OVERLAP
# ============================================================

def intervals_overlap(
    start1,
    end1,
    start2,
    end2
):
    """
    Returns True if two time intervals overlap.
    """

    return (
        start1 < end2
        and
        end1 > start2
    )



In [18]:

# ============================================================
# 5. CHECK TEACHER CONFLICT
# ============================================================

def check_teacher_conflict(
    df,
    current_index,
    teacher_code,
    candidate_day,
    candidate_start,
    candidate_end
):

    # All classes taught by this teacher
    teacher_classes = df[
        (
            df["teacher_code"]
            == teacher_code
        )
        &
        (
            df["day_num"]
            == candidate_day
        )
    ]

    # Remove the current class itself
    teacher_classes = teacher_classes[
        teacher_classes.index != current_index
    ]

    # Check overlap
    for _, other in teacher_classes.iterrows():

        if intervals_overlap(
            candidate_start,
            candidate_end,
            other["start_minutes"],
            other["end_minutes"]
        ):
            return 1

    return 0


In [19]:


# ============================================================
# 6. CHECK SECTION CONFLICT
# ============================================================

def check_section_conflict(
    df,
    current_index,
    class_id,
    batch,
    candidate_day,
    candidate_start,
    candidate_end
):

    # Same class/section
    section_classes = df[
        (
            df["class_id"]
            == class_id
        )
        &
        (
            df["day_num"]
            == candidate_day
        )
    ]

    # Remove current class
    section_classes = section_classes[
        section_classes.index != current_index
    ]

    # --------------------------------------------------------
    # Check each class in the same section
    # --------------------------------------------------------

    for _, other in section_classes.iterrows():

        other_batch = other["batch"]

        # ----------------------------------------------------
        # If BOTH have batches:
        #
        # X vs Y = different student groups
        # Therefore they are NOT a section conflict.
        # ----------------------------------------------------

        if (
            pd.notna(batch)
            and
            pd.notna(other_batch)
        ):

            if batch != other_batch:
                continue

        # ----------------------------------------------------
        # If one has no batch:
        #
        # It represents the whole section.
        #
        # Therefore overlapping with a batched class
        # IS a conflict.
        # ----------------------------------------------------

        if intervals_overlap(
            candidate_start,
            candidate_end,
            other["start_minutes"],
            other["end_minutes"]
        ):
            return 1

    return 0



In [20]:

# ============================================================
# 7. GENERATE CANDIDATE SLOTS
# ============================================================

def generate_candidate_samples(df):

    candidates = []

    # --------------------------------------------------------
    # Process every historical class
    # --------------------------------------------------------

    for index, row in df.iterrows():

        actual_day = row["day_num"]

        actual_start = row["start_minutes"]

        duration_minutes = int(
            row["duration_hours"] * 60
        )

        # ----------------------------------------------------
        # Try every possible day
        # ----------------------------------------------------

        for candidate_day in POSSIBLE_DAYS:

            # ------------------------------------------------
            # Try every possible starting time
            # ------------------------------------------------

            for candidate_start in POSSIBLE_TIMES:

                candidate_end = (
                    candidate_start
                    + duration_minutes
                )

                # ------------------------------------------------
                # Create candidate from original row
                # ------------------------------------------------

                candidate = row.copy()

                # ------------------------------------------------
                # Replace day/time with candidate values
                # ------------------------------------------------

                candidate["day_num"] = candidate_day

                candidate["start_minutes"] = (
                    candidate_start
                )

                candidate["end_minutes"] = (
                    candidate_end
                )

                # ------------------------------------------------
                # Update morning flag
                # ------------------------------------------------

                candidate["is_morning"] = (
                    1
                    if candidate_start < 12 * 60
                    else 0
                )

                # ------------------------------------------------
                # Determine whether this was the actual
                # historical slot
                # ------------------------------------------------

                if (
                    candidate_day == actual_day
                    and
                    candidate_start == actual_start
                ):

                    candidate["is_preferred"] = 1

                else:

                    candidate["is_preferred"] = 0

                # ------------------------------------------------
                # Teacher conflict
                # ------------------------------------------------

                teacher_conflict = (
                    check_teacher_conflict(
                        df=df,
                        current_index=index,
                        teacher_code=row["teacher_code"],
                        candidate_day=candidate_day,
                        candidate_start=candidate_start,
                        candidate_end=candidate_end
                    )
                )

                # ------------------------------------------------
                # Section conflict
                # ------------------------------------------------

                section_conflict = (
                    check_section_conflict(
                        df=df,
                        current_index=index,
                        class_id=row["class_id"],
                        batch=row["batch"],
                        candidate_day=candidate_day,
                        candidate_start=candidate_start,
                        candidate_end=candidate_end
                    )
                )

                # ------------------------------------------------
                # Store conflict flags
                # ------------------------------------------------

                candidate["teacher_conflict"] = (
                    teacher_conflict
                )

                candidate["section_conflict"] = (
                    section_conflict
                )

                # ------------------------------------------------
                # Candidate is feasible only if there is
                # no hard conflict.
                # ------------------------------------------------

                candidate["is_feasible"] = int(
                    teacher_conflict == 0
                    and
                    section_conflict == 0
                )

                # ------------------------------------------------
                # Add candidate
                # ------------------------------------------------

                candidates.append(candidate)

    # Convert list to DataFrame

    candidate_df = pd.DataFrame(
        candidates
    )

    return candidate_df



In [21]:

def add_ml_features(candidate_df):

    df = candidate_df.copy()

    # --------------------------------------------------
    # 1. SLOT INDEX
    # --------------------------------------------------

    SLOT_MAP = {
        585: 0,   # 09:45
        645: 1,   # 10:45
        750: 2,   # 12:30
        810: 3,   # 13:30
        870: 4    # 14:30
    }

    df["slot_index"] = df["start_minutes"].map(SLOT_MAP)


    # --------------------------------------------------
    # 2. TIME FEATURES
    # --------------------------------------------------

    df["is_morning"] = (
        df["start_minutes"] < 720
    ).astype(int)

    df["is_afternoon"] = (
        df["start_minutes"] >= 720
    ).astype(int)


    # --------------------------------------------------
    # 3. DAY FEATURES
    # --------------------------------------------------

    df["is_saturday"] = (
        df["day_num"] == 5
    ).astype(int)

    df["is_week_start"] = (
        df["day_num"] <= 1
    ).astype(int)


    # --------------------------------------------------
    # 4. LOAD FEATURES
    # --------------------------------------------------

    df["teacher_load_ratio"] = (
        df["teacher_daily_load"]
        / (df["teacher_weekly_load"] + 1)
    )

    df["section_load_ratio"] = (
        df["section_daily_load"]
        / (df["section_weekly_load"] + 1)
    )


    # --------------------------------------------------
    # 5. CONFLICT FEATURES
    # --------------------------------------------------

    df["total_conflicts"] = (
        df["teacher_conflict"]
        + df["section_conflict"]
    )


    # --------------------------------------------------
    # 6. FEASIBILITY
    # --------------------------------------------------

    df["is_feasible_numeric"] = (
        df["is_feasible"]
    ).astype(int)


    return df



In [22]:

# ============================================================
# 8. MAIN EXECUTION
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Path to your processed historical JSON
    # --------------------------------------------------------

    JSON_PATH = (
        "../dataset/processed/"
        "historical_processed_corrected.json"
    )

    # --------------------------------------------------------
    # STEP 1:
    # Parse historical timetable
    # --------------------------------------------------------

    df_pos = parse_routine_json(
        JSON_PATH
    )

    print("\n")
    print("=" * 60)
    print("PARSED DATA")
    print("=" * 60)

    print(
        "Historical records:",
        len(df_pos)
    )

    print("\nColumns:")

    print(
        df_pos.columns.tolist()
    )

    # --------------------------------------------------------
    # STEP 2:
    # Generate candidate slots
    # --------------------------------------------------------

    candidate_df = (
        generate_candidate_samples(
            df_pos
        )
    )

    print("\n")
    print("=" * 60)
    print("CANDIDATE GENERATION")
    print("=" * 60)

    print(
        "Total candidate records:",
        len(candidate_df)
    )

    # --------------------------------------------------------
    # Expected:
    #
    # 48 historical records
    # ×
    # 6 days
    # ×
    # 5 starting times
    #
    # = 1440 candidates
    # --------------------------------------------------------

    print("\nExpected candidates:")

    print(
        len(df_pos)
        * len(POSSIBLE_DAYS)
        * len(POSSIBLE_TIMES)
    )

    # --------------------------------------------------------
    # Positive / negative distribution
    # --------------------------------------------------------

    print("\n")
    print("Historical selected vs alternatives:")

    print(
        candidate_df[
            "is_preferred"
        ].value_counts()
    )

    # --------------------------------------------------------
    # Feasible / infeasible distribution
    # --------------------------------------------------------

    print("\n")
    print("Feasible vs conflicting candidates:")

    print(
        candidate_df[
            "is_feasible"
        ].value_counts()
    )

    # --------------------------------------------------------
    # Conflict counts
    # --------------------------------------------------------

    print("\n")
    print("Teacher conflicts:")

    print(
        candidate_df[
            "teacher_conflict"
        ].sum()
    )

    print("\nSection conflicts:")

    print(
        candidate_df[
            "section_conflict"
        ].sum()
    )

    # --------------------------------------------------------
    # Show sample candidates
    # --------------------------------------------------------

    print("\n")
    print("=" * 60)
    print("SAMPLE CANDIDATES")
    print("=" * 60)

    columns_to_show = [
        "discipline",
        "section",
        "batch",
        "subject_id",
        "day_num",
        "start_minutes",
        "end_minutes",
        "duration_hours",
        "record_type",
        "is_preferred",
        "teacher_conflict",
        "section_conflict",
        "is_feasible"
    ]

    print(
        candidate_df[
            columns_to_show
        ].head(30).to_string(
            index=False
        )
    )
    candidate_df = add_ml_features(candidate_df)
    print("\n" + "="*60)
    print("ML FEATURES")
    print("="*60)

    print(candidate_df.head())

    print("\nColumns:")
    print(candidate_df.columns.tolist())

Original records: 48
Records after removing overlaps: 48


PARSED DATA
Historical records: 48

Columns:
['source_file', 'academic_year', 'semester', 'discipline', 'section', 'batch', 'day', 'subject_id', 'subject_name', 'teacher_id', 'teacher_code', 'room', 'start_time', 'end_time', 'duration_hours', 'record_type', 'duplicate_record', 'missing_subject', 'missing_teacher', 'invalid_time_range', 'invalid_duration', 'overlaps_with_another_class', 'start_minutes', 'end_minutes', 'day_num', 'is_lab', 'is_theory', 'is_remedial', 'has_batch', 'is_morning', 'class_id', 'teacher_daily_load', 'teacher_weekly_load', 'section_daily_load', 'section_weekly_load', 'is_preferred']


CANDIDATE GENERATION
Total candidate records: 1440

Expected candidates:
1440


Historical selected vs alternatives:
is_preferred
0    1392
1      48
Name: count, dtype: int64


Feasible vs conflicting candidates:
is_feasible
1    1155
0     285
Name: count, dtype: int64


Teacher conflicts:
79

Section conflicts:
245


SA

In [24]:
# ============================================================
# PREPARE DATA FOR MACHINE LEARNING
# ============================================================

print("\n")
print("=" * 60)
print("PREPARING ML DATASET")
print("=" * 60)

# Features that the Random Forest will actually use
feature_cols = [
    "day_num",
    "start_minutes",
    "duration_hours",
    "is_lab",
    "is_theory",
    "is_remedial",
    "has_batch",
    "is_morning",
    "is_afternoon",
    "is_saturday",
    "is_week_start",
    "teacher_load_ratio",
    "section_load_ratio",
    "teacher_conflict",
    "section_conflict",
    "total_conflicts",
    "is_feasible_numeric"
]

target_col = "is_preferred"

# Create X and y
X = candidate_df[feature_cols].copy()
y = candidate_df[target_col].copy()

print("\nML Features:")
print(feature_cols)

print("\nFeature matrix shape:")
print(X.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nPositive samples:", (y == 1).sum())
print("Negative samples:", (y == 0).sum())



PREPARING ML DATASET

ML Features:
['day_num', 'start_minutes', 'duration_hours', 'is_lab', 'is_theory', 'is_remedial', 'has_batch', 'is_morning', 'is_afternoon', 'is_saturday', 'is_week_start', 'teacher_load_ratio', 'section_load_ratio', 'teacher_conflict', 'section_conflict', 'total_conflicts', 'is_feasible_numeric']

Feature matrix shape:
(1440, 17)

Target distribution:
is_preferred
0    1392
1      48
Name: count, dtype: int64

Positive samples: 48
Negative samples: 1392


In [25]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n")
print("=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())



TRAIN / TEST SPLIT
Training samples: 1152
Testing samples : 288

Training target distribution:
is_preferred
0    1114
1      38
Name: count, dtype: int64

Testing target distribution:
is_preferred
0    278
1     10
Name: count, dtype: int64


In [26]:
# ============================================================
# RANDOM FOREST CLASSIFIER
# ============================================================

from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train)

print("\n")
print("=" * 60)
print("MODEL TRAINING COMPLETE")
print("=" * 60)
print("Random Forest trained successfully.")



MODEL TRAINING COMPLETE
Random Forest trained successfully.


In [27]:
# ============================================================
# MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Make predictions
y_pred = clf.predict(X_test)

print("\n")
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Precision
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

# Recall
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

# F1 Score
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

# Confusion Matrix
print("\nCONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification Report
print("\nCLASSIFICATION REPORT:")
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)



MODEL EVALUATION
Accuracy : 0.8854
Precision: 0.1892
Recall   : 0.7000
F1-Score : 0.2979

CONFUSION MATRIX:
[[248  30]
 [  3   7]]

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.99      0.89      0.94       278
           1       0.19      0.70      0.30        10

    accuracy                           0.89       288
   macro avg       0.59      0.80      0.62       288
weighted avg       0.96      0.89      0.92       288



In [28]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": clf.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print("\n")
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

print(
    importance_df.to_string(index=False)
)



FEATURE IMPORTANCE
            feature  importance
      start_minutes    0.211179
            day_num    0.175361
 section_load_ratio    0.139456
is_feasible_numeric    0.072120
    total_conflicts    0.070962
       is_afternoon    0.065572
         is_morning    0.054102
   section_conflict    0.046294
        is_saturday    0.040711
 teacher_load_ratio    0.039304
      is_week_start    0.019354
          has_batch    0.013879
          is_theory    0.013130
             is_lab    0.012179
     duration_hours    0.010591
        is_remedial    0.009170
   teacher_conflict    0.006637


In [29]:
def minutes_to_time(minutes):

    hours = minutes // 60
    mins = minutes % 60

    suffix = "AM"

    if hours >= 12:
        suffix = "PM"

    display_hour = hours

    if display_hour == 0:
        display_hour = 12

    elif display_hour > 12:
        display_hour -= 12

    return f"{display_hour}:{mins:02d} {suffix}"

In [30]:
# ============================================================
# GENERATE PREFERENCE SCORES
# ============================================================

# Probability that each candidate belongs to class 1
# (preferred slot)

candidate_scores = clf.predict_proba(X)[:, 1]

df_candidates_scored = candidate_df.copy()

df_candidates_scored["preference_score"] = candidate_scores

# ------------------------------------------------------------
# Convert minutes to human-readable time
# ------------------------------------------------------------

df_candidates_scored["start_time_display"] = (
    df_candidates_scored["start_minutes"]
    .apply(minutes_to_time)
)

df_candidates_scored["end_time_display"] = (
    df_candidates_scored["end_minutes"]
    .apply(minutes_to_time)
)

print("\n")
print("=" * 60)
print("PREFERENCE SCORES GENERATED")
print("=" * 60)

print(
    df_candidates_scored[
        [
            "discipline",
            "section",
            "subject_id",
            "day_num",
            "start_time_display",
            "end_time_display",
            "is_preferred",
            "is_feasible",
            "preference_score"
        ]
    ]
    .head(20)
    .to_string(index=False)
)



PREFERENCE SCORES GENERATED
discipline section subject_id  day_num start_time_display end_time_display  is_preferred  is_feasible  preference_score
      CSBS    CSBS   BS-CH201        0            9:45 AM         10:45 AM             0            1          0.118365
      CSBS    CSBS   BS-CH201        0           10:45 AM         11:45 AM             0            1          0.041775
      CSBS    CSBS   BS-CH201        0           12:30 PM          1:30 PM             0            1          0.006401
      CSBS    CSBS   BS-CH201        0            1:30 PM          2:30 PM             0            1          0.007243
      CSBS    CSBS   BS-CH201        0            2:30 PM          3:30 PM             0            1          0.000000
      CSBS    CSBS   BS-CH201        1            9:45 AM         10:45 AM             1            1          0.940636
      CSBS    CSBS   BS-CH201        1           10:45 AM         11:45 AM             0            0          0.006239
      CSBS

In [71]:
# ============================================================
# RECURSIVE TIMETABLE GENERATION
# ============================================================

def generate_timetable(candidate_df):
    """
    Generates a timetable using ML preference scores
    with recursive backtracking.

    The candidate with the highest preference score is
    always tried first.

    If that candidate causes a conflict later, the algorithm
    backtracks and tries the next-highest-scoring candidate.

    Hard constraints:
        1. Teacher cannot have overlapping classes.
        2. Section cannot have overlapping classes.
        3. Room cannot have overlapping classes.
        4. A class can only be scheduled once.
    """

    # ========================================================
    # 1. KEEP ONLY FEASIBLE CANDIDATES
    # ========================================================

    feasible = candidate_df[
        candidate_df["is_feasible"] == 1
    ].copy()

    # ========================================================
    # 2. REMOVE EXACT DUPLICATES
    # ========================================================

    feasible = feasible.drop_duplicates(
        subset=[
            "discipline",
            "section",
            "subject_id",
            "day_num",
            "start_minutes",
            "end_minutes",
            "teacher_code",
            "room"
        ]
    )

    # ========================================================
    # 3. GROUP CANDIDATES BY CLASS
    # ========================================================

    class_columns = [
        "discipline",
        "section",
        "subject_id"
    ]

    grouped = list(
        feasible.groupby(
            class_columns,
            sort=False
        )
    )

    # ========================================================
    # 4. SORT CANDIDATES OF EACH CLASS
    #    HIGHEST SCORE FIRST
    # ========================================================

    class_candidates = []

    for class_key, group in grouped:

        group = group.sort_values(
            "preference_score",
            ascending=False
        ).reset_index(drop=True)

        class_candidates.append(
            (
                class_key,
                group
            )
        )

    # ========================================================
    # 5. PROCESS MORE RESTRICTED CLASSES FIRST
    # ========================================================
    #
    # Classes having fewer feasible candidates are harder
    # to schedule, so schedule them first.
    #
    # If two classes have the same number of candidates,
    # keep their original order.
    # ========================================================

    class_candidates.sort(
        key=lambda x: len(x[1])
    )

    # ========================================================
    # 6. RESOURCE TRACKING
    # ========================================================

    teacher_schedule = set()
    section_schedule = set()
    room_schedule = set()

    selected = []

    # ========================================================
    # 7. CHECK WHETHER A CANDIDATE IS VALID
    # ========================================================

    def is_valid_candidate(candidate):

        teacher = candidate["teacher_code"]
        section = candidate["section"]
        room = candidate["room"]

        day = candidate["day_num"]
        start = candidate["start_minutes"]
        end = candidate["end_minutes"]

        # ----------------------------------------------------
        # TEACHER CONFLICT
        # ----------------------------------------------------

        if not pd.isna(teacher):

            for (
                existing_teacher,
                existing_day,
                existing_start,
                existing_end
            ) in teacher_schedule:

                if (
                    existing_teacher == teacher
                    and existing_day == day
                ):

                    if (
                        start < existing_end
                        and
                        end > existing_start
                    ):

                        return False

        # ----------------------------------------------------
        # SECTION CONFLICT
        # ----------------------------------------------------

        for (
            existing_section,
            existing_day,
            existing_start,
            existing_end
        ) in section_schedule:

            if (
                existing_section == section
                and existing_day == day
            ):

                if (
                    start < existing_end
                    and
                    end > existing_start
                ):

                    return False

        # ----------------------------------------------------
        # ROOM CONFLICT
        # ----------------------------------------------------

        if not pd.isna(room) and str(room).strip() != "":

            for (
                existing_room,
                existing_day,
                existing_start,
                existing_end
            ) in room_schedule:

                if (
                    existing_room == room
                    and existing_day == day
                ):

                    if (
                        start < existing_end
                        and
                        end > existing_start
                    ):

                        return False

        # ----------------------------------------------------
        # ALL CONSTRAINTS PASSED
        # ----------------------------------------------------

        return True

    # ========================================================
    # 8. ADD CANDIDATE TO CURRENT SCHEDULE
    # ========================================================

    def add_candidate(candidate):

        teacher = candidate["teacher_code"]
        section = candidate["section"]
        room = candidate["room"]

        day = candidate["day_num"]
        start = candidate["start_minutes"]
        end = candidate["end_minutes"]

        # ----------------------------------------------------
        # Teacher
        # ----------------------------------------------------

        if not pd.isna(teacher):

            teacher_schedule.add(
                (
                    teacher,
                    day,
                    start,
                    end
                )
            )

        # ----------------------------------------------------
        # Section
        # ----------------------------------------------------

        section_schedule.add(
            (
                section,
                day,
                start,
                end
            )
        )

        # ----------------------------------------------------
        # Room
        # ----------------------------------------------------

        if not pd.isna(room) and str(room).strip() != "":

            room_schedule.add(
                (
                    room,
                    day,
                    start,
                    end
                )
            )

        # ----------------------------------------------------
        # Selected timetable
        # ----------------------------------------------------

        selected.append(candidate)

    # ========================================================
    # 9. REMOVE CANDIDATE FROM CURRENT SCHEDULE
    # ========================================================

    def remove_candidate(candidate):

        teacher = candidate["teacher_code"]
        section = candidate["section"]
        room = candidate["room"]

        day = candidate["day_num"]
        start = candidate["start_minutes"]
        end = candidate["end_minutes"]

        # ----------------------------------------------------
        # Teacher
        # ----------------------------------------------------

        if not pd.isna(teacher):

            teacher_schedule.discard(
                (
                    teacher,
                    day,
                    start,
                    end
                )
            )

        # ----------------------------------------------------
        # Section
        # ----------------------------------------------------

        section_schedule.discard(
            (
                section,
                day,
                start,
                end
            )
        )

        # ----------------------------------------------------
        # Room
        # ----------------------------------------------------

        if not pd.isna(room) and str(room).strip() != "":

            room_schedule.discard(
                (
                    room,
                    day,
                    start,
                    end
                )
            )

        # ----------------------------------------------------
        # Selected timetable
        # ----------------------------------------------------

        selected.pop()

    # ========================================================
    # 10. RECURSIVE BACKTRACKING
    # ========================================================

    def backtrack(class_index):

        # ----------------------------------------------------
        # BASE CASE
        # ----------------------------------------------------
        #
        # All classes have been successfully scheduled.
        # ----------------------------------------------------

        if class_index == len(class_candidates):

            return True

        class_key, candidates = class_candidates[class_index]

        # ----------------------------------------------------
        # TRY CANDIDATES IN PREFERENCE ORDER
        # ----------------------------------------------------

        for _, candidate in candidates.iterrows():

            # ------------------------------------------------
            # Check hard constraints
            # ------------------------------------------------

            if not is_valid_candidate(candidate):

                continue

            # ------------------------------------------------
            # Select candidate
            # ------------------------------------------------

            add_candidate(candidate)

            # ------------------------------------------------
            # Recursively schedule next class
            # ------------------------------------------------

            success = backtrack(
                class_index + 1
            )

            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            if success:

                return True

            # ------------------------------------------------
            # FAILURE
            #
            # This candidate caused a problem later.
            #
            # Undo it and try the next-highest candidate.
            # ------------------------------------------------

            remove_candidate(candidate)

        # ----------------------------------------------------
        # No candidate worked for this class.
        #
        # Tell the previous recursion level to backtrack.
        # ----------------------------------------------------

        return False

    # ========================================================
    # 11. START RECURSIVE SCHEDULING
    # ========================================================

    print("\n")
    print("=" * 60)
    print("STARTING RECURSIVE TIMETABLE GENERATION")
    print("=" * 60)

    success = backtrack(0)

    # ========================================================
    # 12. GENERATION RESULT
    # ========================================================

    if success:

        print("\n✓ Recursive scheduling completed successfully.")

    else:

        print("\n✗ Could not schedule all classes.")

    # ========================================================
    # 13. CREATE DATAFRAME
    # ========================================================

    timetable = pd.DataFrame(selected)

    # ========================================================
    # 14. ADD DISPLAY TIME COLUMNS
    # ========================================================

    if not timetable.empty:

        timetable["start_time_display"] = (
            timetable["start_minutes"]
            .apply(minutes_to_time)
        )

        timetable["end_time_display"] = (
            timetable["end_minutes"]
            .apply(minutes_to_time)
        )

    return timetable

In [72]:
# ============================================================
# GENERATE FINAL TIMETABLE
# ============================================================

final_timetable = generate_timetable(
    df_candidates_scored
)

print("\n")
print("=" * 60)
print("FINAL GENERATED TIMETABLE")
print("=" * 60)

print(
    final_timetable[
        [
            "discipline",
            "section",
            "subject_id",
            "day_num",
            "start_time_display",
            "end_time_display",
            "teacher_code",
            "room",
            "record_type",
            "preference_score"
        ]
    ].sort_values(
        [
            "section",
            "day_num",
            "start_time_display"
        ]
    ).to_string(index=False)
)

# ============================================================
# SCHEDULING COVERAGE
# ============================================================

required_classes = (
    candidate_df[
        [
            "discipline",
            "section",
            "subject_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

scheduled_classes = (
    final_timetable[
        [
            "discipline",
            "section",
            "subject_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\n")
print("=" * 60)
print("SCHEDULING COVERAGE")
print("=" * 60)

print(
    f"Required classes : {required_classes}"
)

print(
    f"Scheduled classes: {scheduled_classes}"
)

print(
    f"Unscheduled       : "
    f"{required_classes - scheduled_classes}"
)



STARTING RECURSIVE TIMETABLE GENERATION

✓ Recursive scheduling completed successfully.


FINAL GENERATED TIMETABLE
discipline section subject_id  day_num start_time_display end_time_display  teacher_code room record_type  preference_score
      CSBS    CSBS   HM-HU291        1           10:45 AM         11:45 AM            SS  NaN      theory          0.856477
      CSBS    CSBS   BS-CH201        1            9:45 AM         10:45 AM           ABM  NaN      theory          0.940636
      CSBS    CSBS   BS-CH291        4            9:45 AM         11:45 AM            PJ  NaN         lab          0.934642
      CSBS    CSBS   ES-CS291        5            9:45 AM         11:45 AM            SD  L-3         lab          0.681986
       CSE    CSE1   BS-CH201        1           10:45 AM         11:45 AM            DG  NaN      theory          0.834962
       CSE    CSE1   HM-HU201        1           12:30 PM          1:30 PM            MM  NaN      theory          0.858738
       CSE    

In [73]:
# ============================================================
# FINAL TIMETABLE VALIDATION
# ============================================================

def validate_timetable(timetable):

    print("\n")
    print("=" * 60)
    print("FINAL TIMETABLE VALIDATION")
    print("=" * 60)

    # Make a copy so original timetable is not modified
    timetable = timetable.copy()

    duplicate_classes = []
    teacher_conflicts = []
    section_conflicts = []
    room_conflicts = []

    # --------------------------------------------------------
    # Sort timetable
    # --------------------------------------------------------

    timetable = timetable.sort_values(
        [
            "day_num",
            "start_minutes"
        ]
    ).reset_index(drop=True)

    # ========================================================
    # 0. DUPLICATE CLASS CHECK
    # ========================================================

    # A subject should not be scheduled more times than it
    # appears in the required timetable.
    #
    # We first check exact duplicate class assignments.
    # discipline + section + subject_id identifies a class.

    class_columns = [
        "discipline",
        "section",
        "subject_id"
    ]

    class_counts = (
        timetable
        .groupby(class_columns, dropna=False)
        .size()
        .reset_index(name="count")
    )

    duplicate_classes = class_counts[
        class_counts["count"] > 1
    ].to_dict("records")

    print("\nDuplicate class assignments:", len(duplicate_classes))

    if duplicate_classes:

        print("\n")
        print("-" * 60)
        print("DUPLICATE CLASS DETAILS")
        print("-" * 60)

        for dup in duplicate_classes:

            print(
                f"{dup['discipline']} "
                f"{dup['section']} "
                f"{dup['subject_id']} "
                f"appears {dup['count']} times"
            )

    # ========================================================
    # 1. TEACHER CONFLICTS
    # ========================================================

    for i in range(len(timetable)):

        row1 = timetable.iloc[i]

        # Ignore missing teacher
        if pd.isna(row1["teacher_code"]):
            continue

        for j in range(i + 1, len(timetable)):

            row2 = timetable.iloc[j]

            # Ignore missing teacher
            if pd.isna(row2["teacher_code"]):
                continue

            # Same teacher + same day
            if (
                row1["teacher_code"]
                == row2["teacher_code"]
                and
                row1["day_num"]
                == row2["day_num"]
            ):

                # Time overlap
                if (
                    row1["start_minutes"] < row2["end_minutes"]
                    and
                    row1["end_minutes"] > row2["start_minutes"]
                ):

                    teacher_conflicts.append(
                        (
                            row1["teacher_code"],
                            row1["subject_id"],
                            row2["subject_id"],
                            row1["day"],
                            row1["start_time"],
                            row1["end_time"],
                            row2["start_time"],
                            row2["end_time"]
                        )
                    )

    # ========================================================
    # 2. SECTION CONFLICTS
    # ========================================================

    for i in range(len(timetable)):

        row1 = timetable.iloc[i]

        for j in range(i + 1, len(timetable)):

            row2 = timetable.iloc[j]

            # Same section + same day
            if (
                row1["section"]
                == row2["section"]
                and
                row1["day_num"]
                == row2["day_num"]
            ):

                # Time overlap
                if (
                    row1["start_minutes"] < row2["end_minutes"]
                    and
                    row1["end_minutes"] > row2["start_minutes"]
                ):

                    section_conflicts.append(
                        (
                            row1["section"],
                            row1["subject_id"],
                            row2["subject_id"],
                            row1["day"],
                            row1["start_time"],
                            row1["end_time"],
                            row2["start_time"],
                            row2["end_time"]
                        )
                    )

    # ========================================================
    # 3. ROOM CONFLICTS
    # ========================================================

    for i in range(len(timetable)):

        row1 = timetable.iloc[i]

        # Ignore missing rooms
        if pd.isna(row1["room"]):
            continue

        # Ignore empty room strings
        if str(row1["room"]).strip() == "":
            continue

        for j in range(i + 1, len(timetable)):

            row2 = timetable.iloc[j]

            # Ignore missing rooms
            if pd.isna(row2["room"]):
                continue

            # Ignore empty room strings
            if str(row2["room"]).strip() == "":
                continue

            # Same room + same day
            if (
                str(row1["room"]).strip()
                ==
                str(row2["room"]).strip()
                and
                row1["day_num"]
                ==
                row2["day_num"]
            ):

                # Time overlap
                if (
                    row1["start_minutes"] < row2["end_minutes"]
                    and
                    row1["end_minutes"] > row2["start_minutes"]
                ):

                    room_conflicts.append(
                        (
                            row1["room"],
                            row1["subject_id"],
                            row2["subject_id"],
                            row1["day"],
                            row1["start_time"],
                            row1["end_time"],
                            row2["start_time"],
                            row2["end_time"]
                        )
                    )

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print("\nDuplicate class assignments:", len(duplicate_classes))

    print("Teacher conflicts:", len(teacher_conflicts))

    print("Section conflicts:", len(section_conflicts))

    print("Room conflicts:", len(room_conflicts))

    # ========================================================
    # DUPLICATE CLASS DETAILS
    # ========================================================

    if duplicate_classes:

        print("\n")
        print("-" * 60)
        print("DUPLICATE CLASS DETAILS")
        print("-" * 60)

        for dup in duplicate_classes:

            print(
                f"{dup['discipline']} "
                f"{dup['section']} "
                f"{dup['subject_id']} "
                f"appears {dup['count']} times"
            )

    # ========================================================
    # TEACHER CONFLICT DETAILS
    # ========================================================

    if teacher_conflicts:

        print("\n")
        print("-" * 60)
        print("TEACHER CONFLICT DETAILS")
        print("-" * 60)

        for conflict in teacher_conflicts:

            print(
                f"Teacher {conflict[0]}: "
                f"{conflict[1]} "
                f"({conflict[4]}-{conflict[5]}) "
                f"OVERLAPS WITH "
                f"{conflict[2]} "
                f"({conflict[6]}-{conflict[7]}) "
                f"on {conflict[3]}"
            )

    # ========================================================
    # SECTION CONFLICT DETAILS
    # ========================================================

    if section_conflicts:

        print("\n")
        print("-" * 60)
        print("SECTION CONFLICT DETAILS")
        print("-" * 60)

        for conflict in section_conflicts:

            print(
                f"Section {conflict[0]}: "
                f"{conflict[1]} "
                f"({conflict[4]}-{conflict[5]}) "
                f"OVERLAPS WITH "
                f"{conflict[2]} "
                f"({conflict[6]}-{conflict[7]}) "
                f"on {conflict[3]}"
            )

    # ========================================================
    # ROOM CONFLICT DETAILS
    # ========================================================

    if room_conflicts:

        print("\n")
        print("-" * 60)
        print("ROOM CONFLICT DETAILS")
        print("-" * 60)

        for conflict in room_conflicts:

            print(
                f"Room {conflict[0]}: "
                f"{conflict[1]} "
                f"({conflict[4]}-{conflict[5]}) "
                f"OVERLAPS WITH "
                f"{conflict[2]} "
                f"({conflict[6]}-{conflict[7]}) "
                f"on {conflict[3]}"
            )

    # ========================================================
    # FINAL STATUS
    # ========================================================

    if (
        len(duplicate_classes) == 0
        and
        len(teacher_conflicts) == 0
        and
        len(section_conflicts) == 0
        and
        len(room_conflicts) == 0
    ):

        print("\n")
        print("=" * 60)
        print("✓ TIMETABLE IS VALID")
        print("=" * 60)

    else:

        print("\n")
        print("=" * 60)
        print("✗ TIMETABLE HAS CONFLICTS")
        print("=" * 60)

    # ========================================================
    # RETURN VALIDATION RESULTS
    # ========================================================

    return {
        "duplicate_classes": duplicate_classes,
        "teacher_conflicts": teacher_conflicts,
        "section_conflicts": section_conflicts,
        "room_conflicts": room_conflicts
    }

In [74]:
# ============================================================
# RUN VALIDATION
# ============================================================

validation_result = validate_timetable(
    final_timetable
)



FINAL TIMETABLE VALIDATION

Duplicate class assignments: 0

Duplicate class assignments: 0
Teacher conflicts: 0
Section conflicts: 0
Room conflicts: 0


✓ TIMETABLE IS VALID


In [75]:
joblib.dump(clf, "random_forest_model.pkl")

print("Model saved successfully.")

Model saved successfully.


## Testing

In [76]:
import json
import pandas as pd
import numpy as np
import joblib


# ============================================================
# LOAD TRAINED MODEL
# ============================================================

clf = joblib.load("random_forest_model.pkl")

print("Random Forest model loaded successfully.")

Random Forest model loaded successfully.


In [77]:
# ============================================================
# LOAD NEW INPUT DATASET
# ============================================================

JSON_PATH = "../dataset/processed/current_processed.json"

with open(JSON_PATH, "r") as f:
    new_data = json.load(f)

print("\n")
print("=" * 60)
print("NEW DATASET LOADED")
print("=" * 60)

print("Subjects:",
      len(new_data.get("subjects", [])))

print("Teacher qualifications:",
      len(new_data.get("teacher_qualifications", [])))

print("Time slots:",
      len(new_data.get("time_slots", [])))



NEW DATASET LOADED
Subjects: 7
Teacher qualifications: 15
Time slots: 42


In [78]:
# ============================================================
# PARSE SUBJECTS
# ============================================================

subjects = pd.DataFrame(
    new_data["subjects"]
)

print("\n")
print("=" * 60)
print("SUBJECTS")
print("=" * 60)

print(
    subjects[
        [
            "subject_id",
            "subject_name",
            "subject_type",
            "weekly_periods",
            "consecutive_periods"
        ]
    ].to_string(index=False)
)



SUBJECTS
subject_id     subject_name subject_type  weekly_periods  consecutive_periods
 PCC-CS501   Core Subject 1       theory               4                  NaN
 PCC-CS502   Core Subject 2       theory               4                  NaN
 PCC-CS503   Core Subject 3       theory               4                  NaN
   HSMC501             HSMC       theory               3                  NaN
PEC-IT501B Program Elective       theory               3                  NaN
 PCC-CS592       Core Lab 1          lab               3                  2.0
 PCC-CS593       Core Lab 2          lab               2                  2.0


In [79]:
# ============================================================
# PARSE TEACHER QUALIFICATIONS
# ============================================================

teacher_qualifications = pd.DataFrame(
    new_data["teacher_qualifications"]
)

print("\n")
print("=" * 60)
print("TEACHER QUALIFICATIONS")
print("=" * 60)

print(
    teacher_qualifications[
        [
            "teacher_id",
            "teacher_code",
            "teacher_name",
            "subject_id",
            "can_teach_theory",
            "can_teach_lab",
            "max_subjects"
        ]
    ].to_string(index=False)
)



TEACHER QUALIFICATIONS
teacher_id teacher_code   teacher_name subject_id  can_teach_theory  can_teach_lab  max_subjects
      T001          SBR Test Teacher A  PCC-CS501              True          False             2
      T001          SBR Test Teacher A  PCC-CS503              True          False             2
      T002           PD Test Teacher B  PCC-CS501              True           True             3
      T002           PD Test Teacher B  PCC-CS503              True           True             3
      T002           PD Test Teacher B  PCC-CS592              True           True             3
      T003           AP Test Teacher C  PCC-CS502              True           True             2
      T003           AP Test Teacher C  PCC-CS592              True           True             2
      T004           SH Test Teacher D  PCC-CS502              True           True             3
      T004           SH Test Teacher D  PCC-CS503              True           True             3
     

In [80]:
# ============================================================
# PARSE TIME SLOTS
# ============================================================

time_slots = pd.DataFrame(
    new_data["time_slots"]
)

# Remove break
time_slots = time_slots[
    time_slots["is_break"] == False
].copy()


def time_to_minutes(time_str):

    h, m = map(
        int,
        time_str.split(":")
    )

    return h * 60 + m


time_slots["start_minutes"] = (
    time_slots["start_time"]
    .apply(time_to_minutes)
)

time_slots["end_minutes"] = (
    time_slots["end_time"]
    .apply(time_to_minutes)
)


DAY_MAP = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5
}


time_slots["day_num"] = (
    time_slots["day"]
    .map(DAY_MAP)
)

print("\n")
print("=" * 60)
print("AVAILABLE TIME SLOTS")
print("=" * 60)

print(
    time_slots[
        [
            "day",
            "slot_id",
            "start_time",
            "end_time",
            "day_num",
            "start_minutes",
            "end_minutes"
        ]
    ].head(20).to_string(index=False)
)



AVAILABLE TIME SLOTS
      day slot_id start_time end_time  day_num  start_minutes  end_minutes
   Monday      S1      09:45    10:45        0            585          645
   Monday      S2      10:45    11:45        0            645          705
   Monday      S3      11:45    12:45        0            705          765
   Monday      S4      13:30    14:30        0            810          870
   Monday      S5      14:30    15:30        0            870          930
   Monday      S6      15:30    16:30        0            930          990
  Tuesday      S1      09:45    10:45        1            585          645
  Tuesday      S2      10:45    11:45        1            645          705
  Tuesday      S3      11:45    12:45        1            705          765
  Tuesday      S4      13:30    14:30        1            810          870
  Tuesday      S5      14:30    15:30        1            870          930
  Tuesday      S6      15:30    16:30        1            930          990
We

In [81]:
# ============================================================
# TEACHER QUALIFICATION MAP
# ============================================================

teacher_map = {}

for _, row in teacher_qualifications.iterrows():

    subject_id = row["subject_id"]

    if subject_id not in teacher_map:
        teacher_map[subject_id] = []

    teacher_map[subject_id].append(
        {
            "teacher_id": row["teacher_id"],
            "teacher_code": row["teacher_code"],
            "teacher_name": row["teacher_name"],
            "can_teach_theory": row["can_teach_theory"],
            "can_teach_lab": row["can_teach_lab"],
            "max_subjects": row["max_subjects"]
        }
    )


print("\n")
print("=" * 60)
print("TEACHER MAP")
print("=" * 60)

for subject_id, teachers in teacher_map.items():

    print(
        subject_id,
        "->",
        [
            t["teacher_code"]
            for t in teachers
        ]
    )



TEACHER MAP
PCC-CS501 -> ['SBR', 'PD']
PCC-CS503 -> ['SBR', 'PD', 'SH']
PCC-CS592 -> ['PD', 'AP']
PCC-CS502 -> ['AP', 'SH']
PCC-CS593 -> ['SH', 'AM']
HSMC501 -> ['NF(MM)', 'BBH']
PEC-IT501B -> ['BBH', 'OS']


In [82]:
# ============================================================
# GENERATE CANDIDATE ASSIGNMENTS
# ============================================================

candidates = []


for _, subject in subjects.iterrows():

    subject_id = subject["subject_id"]
    subject_type = subject["subject_type"]

    # Teachers qualified for this subject
    qualified_teachers = teacher_map.get(
        subject_id,
        []
    )

    for teacher in qualified_teachers:

        # Check teaching ability
        if subject_type == "theory":

            if not teacher["can_teach_theory"]:
                continue

        elif subject_type == "lab":

            if not teacher["can_teach_lab"]:
                continue

        # Generate candidate for every slot
        for _, slot in time_slots.iterrows():

            candidates.append(
                {
                    "subject_id": subject_id,
                    "subject_name": subject["subject_name"],
                    "record_type": subject_type,

                    "teacher_id":
                        teacher["teacher_id"],

                    "teacher_code":
                        teacher["teacher_code"],

                    "teacher_name":
                        teacher["teacher_name"],

                    "max_subjects":
                        teacher["max_subjects"],

                    "day":
                        slot["day"],

                    "day_num":
                        slot["day_num"],

                    "slot_id":
                        slot["slot_id"],

                    "start_time":
                        slot["start_time"],

                    "end_time":
                        slot["end_time"],

                    "start_minutes":
                        slot["start_minutes"],

                    "end_minutes":
                        slot["end_minutes"],

                    "duration_hours":
                        (
                            slot["end_minutes"]
                            -
                            slot["start_minutes"]
                        ) / 60,

                    "weekly_periods":
                        subject["weekly_periods"],

                    "consecutive_periods":
                        subject["consecutive_periods"]
                }
            )


candidate_df = pd.DataFrame(candidates)


print("\n")
print("=" * 60)
print("CANDIDATE GENERATION")
print("=" * 60)

print(
    "Total candidates:",
    len(candidate_df)
)

print(
    "\nCandidates per subject:"
)

print(
    candidate_df
    .groupby("subject_id")
    .size()
)



CANDIDATE GENERATION
Total candidates: 540

Candidates per subject:
subject_id
HSMC501        72
PCC-CS501      72
PCC-CS502      72
PCC-CS503     108
PCC-CS592      72
PCC-CS593      72
PEC-IT501B     72
dtype: int64


In [83]:
print(candidate_df.columns.tolist())

['subject_id', 'subject_name', 'record_type', 'teacher_id', 'teacher_code', 'teacher_name', 'max_subjects', 'day', 'day_num', 'slot_id', 'start_time', 'end_time', 'start_minutes', 'end_minutes', 'duration_hours', 'weekly_periods', 'consecutive_periods']


In [84]:
print(candidate_df.head().to_string())

  subject_id    subject_name record_type teacher_id teacher_code    teacher_name  max_subjects     day  day_num slot_id start_time end_time  start_minutes  end_minutes  duration_hours  weekly_periods  consecutive_periods
0  PCC-CS501  Core Subject 1      theory       T001          SBR  Test Teacher A             2  Monday        0      S1      09:45    10:45            585          645             1.0               4                  NaN
1  PCC-CS501  Core Subject 1      theory       T001          SBR  Test Teacher A             2  Monday        0      S2      10:45    11:45            645          705             1.0               4                  NaN
2  PCC-CS501  Core Subject 1      theory       T001          SBR  Test Teacher A             2  Monday        0      S3      11:45    12:45            705          765             1.0               4                  NaN
3  PCC-CS501  Core Subject 1      theory       T001          SBR  Test Teacher A             2  Monday        0     

In [85]:
candidate_df.shape

(540, 17)

In [86]:
# ============================================================
# PREPARE ML FEATURES FOR NEW DATASET
# ============================================================

df_new_features = candidate_df.copy()

print("\n")
print("=" * 60)
print("PREPARING ML FEATURES FOR NEW DATASET")
print("=" * 60)

# ------------------------------------------------------------
# 1. Basic class-type features
# ------------------------------------------------------------

df_new_features["is_lab"] = (
    df_new_features["record_type"]
    .astype(str)
    .str.lower()
    .eq("lab")
    .astype(int)
)

df_new_features["is_theory"] = (
    df_new_features["record_type"]
    .astype(str)
    .str.lower()
    .eq("theory")
    .astype(int)
)

# New dataset currently has no remedial/batch information
df_new_features["is_remedial"] = 0
df_new_features["has_batch"] = 0


# ------------------------------------------------------------
# 2. Time-based features
# ------------------------------------------------------------

df_new_features["is_morning"] = (
    df_new_features["start_minutes"] < 750
).astype(int)

df_new_features["is_afternoon"] = (
    df_new_features["start_minutes"] >= 750
).astype(int)


# ------------------------------------------------------------
# 3. Day-based features
# ------------------------------------------------------------

df_new_features["is_saturday"] = (
    df_new_features["day"]
    .astype(str)
    .str.lower()
    .eq("saturday")
    .astype(int)
)

df_new_features["is_week_start"] = (
    df_new_features["day_num"] == 0
).astype(int)


# ============================================================
# 4. TEACHER LOAD
# ============================================================

# Number of candidate classes associated with each teacher
teacher_load = (
    df_new_features
    .groupby("teacher_code")["subject_id"]
    .transform("count")
)

# Avoid division by zero
teacher_load = teacher_load.fillna(0)

# Normalize teacher load
max_teacher_load = teacher_load.max()

if max_teacher_load > 0:
    df_new_features["teacher_load_ratio"] = (
        teacher_load / max_teacher_load
    )
else:
    df_new_features["teacher_load_ratio"] = 0.0


# ============================================================
# 5. SECTION LOAD
# ============================================================

# This dataset does not contain a section column.
# Treat the complete dataset as one scheduling group.

df_new_features["section_load_ratio"] = 0.0


# ============================================================
# 6. TEACHER CONFLICT
# ============================================================

df_new_features["teacher_conflict"] = 0

for i in range(len(df_new_features)):

    row1 = df_new_features.iloc[i]

    if pd.isna(row1["teacher_code"]):
        continue

    for j in range(i + 1, len(df_new_features)):

        row2 = df_new_features.iloc[j]

        if pd.isna(row2["teacher_code"]):
            continue

        if (
            row1["teacher_code"] == row2["teacher_code"]
            and
            row1["day_num"] == row2["day_num"]
        ):

            if (
                row1["start_minutes"] < row2["end_minutes"]
                and
                row1["end_minutes"] > row2["start_minutes"]
            ):

                df_new_features.loc[
                    df_new_features.index[i],
                    "teacher_conflict"
                ] = 1

                df_new_features.loc[
                    df_new_features.index[j],
                    "teacher_conflict"
                ] = 1


# ============================================================
# 7. SECTION CONFLICT
# ============================================================

# There is no section information in this new dataset.
# Therefore there is no section conflict at candidate-generation
# stage.

df_new_features["section_conflict"] = 0


# ============================================================
# 8. TOTAL CONFLICTS
# ============================================================

df_new_features["total_conflicts"] = (
    df_new_features["teacher_conflict"]
    +
    df_new_features["section_conflict"]
)


# ============================================================
# 9. FEASIBILITY
# ============================================================

# Candidate is feasible if it has no teacher/section conflict.

df_new_features["is_feasible_numeric"] = (
    (
        df_new_features["teacher_conflict"] == 0
    )
    &
    (
        df_new_features["section_conflict"] == 0
    )
).astype(int)


# Also create is_feasible because generate_timetable()
# expects this column.

df_new_features["is_feasible"] = (
    df_new_features["is_feasible_numeric"]
)


# ============================================================
# 10. FINAL ML FEATURES
# ============================================================

ml_features = [
    "day_num",
    "start_minutes",
    "duration_hours",
    "is_lab",
    "is_theory",
    "is_remedial",
    "has_batch",
    "is_morning",
    "is_afternoon",
    "is_saturday",
    "is_week_start",
    "teacher_load_ratio",
    "section_load_ratio",
    "teacher_conflict",
    "section_conflict",
    "total_conflicts",
    "is_feasible_numeric"
]


print("\nML Features:")
print(ml_features)

print(
    "\nFeature matrix shape:",
    df_new_features[ml_features].shape
)

print("\nFeature sample:")

print(
    df_new_features[
        ml_features
    ].head().to_string(index=False)
)



PREPARING ML FEATURES FOR NEW DATASET

ML Features:
['day_num', 'start_minutes', 'duration_hours', 'is_lab', 'is_theory', 'is_remedial', 'has_batch', 'is_morning', 'is_afternoon', 'is_saturday', 'is_week_start', 'teacher_load_ratio', 'section_load_ratio', 'teacher_conflict', 'section_conflict', 'total_conflicts', 'is_feasible_numeric']

Feature matrix shape: (540, 17)

Feature sample:
 day_num  start_minutes  duration_hours  is_lab  is_theory  is_remedial  has_batch  is_morning  is_afternoon  is_saturday  is_week_start  teacher_load_ratio  section_load_ratio  teacher_conflict  section_conflict  total_conflicts  is_feasible_numeric
       0            585             1.0       0          1            0          0           1             0            0              1            0.666667                 0.0                 1                 0                1                    0
       0            645             1.0       0          1            0          0           1             0

In [87]:
# ============================================================
# PREDICT PREFERENCE SCORES USING TRAINED MODEL
# ============================================================

X_new = df_new_features[ml_features].copy()

candidate_scores = clf.predict_proba(X_new)[:, 1]

df_candidates_scored_new = df_new_features.copy()

df_candidates_scored_new["preference_score"] = candidate_scores


print("\n")
print("=" * 60)
print("PREFERENCE SCORES GENERATED FOR NEW DATASET")
print("=" * 60)

print(
    df_candidates_scored_new[
        [
            "subject_id",
            "subject_name",
            "teacher_code",
            "day",
            "start_time",
            "end_time",
            "is_feasible",
            "preference_score"
        ]
    ]
    .sort_values(
        "preference_score",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)



PREFERENCE SCORES GENERATED FOR NEW DATASET
subject_id     subject_name teacher_code       day start_time end_time  is_feasible  preference_score
 PCC-CS593       Core Lab 2           AM  Thursday      09:45    10:45            1          0.698455
PEC-IT501B Program Elective           OS   Tuesday      09:45    10:45            1          0.692258
   HSMC501             HSMC       NF(MM)   Tuesday      09:45    10:45            1          0.692258
 PCC-CS593       Core Lab 2           AM    Friday      09:45    10:45            1          0.687105
PEC-IT501B Program Elective           OS  Thursday      09:45    10:45            1          0.681381
   HSMC501             HSMC       NF(MM)  Thursday      09:45    10:45            1          0.681381
   HSMC501             HSMC       NF(MM) Wednesday      09:45    10:45            1          0.675089
PEC-IT501B Program Elective           OS Wednesday      09:45    10:45            1          0.675089
PEC-IT501B Program Elective         

In [97]:
df_candidates_scored_new.columns.tolist()

['subject_id',
 'subject_name',
 'record_type',
 'teacher_id',
 'teacher_code',
 'teacher_name',
 'max_subjects',
 'day',
 'day_num',
 'slot_id',
 'start_time',
 'end_time',
 'start_minutes',
 'end_minutes',
 'duration_hours',
 'weekly_periods',
 'consecutive_periods',
 'is_lab',
 'is_theory',
 'is_remedial',
 'has_batch',
 'is_morning',
 'is_afternoon',
 'is_saturday',
 'is_week_start',
 'teacher_load_ratio',
 'section_load_ratio',
 'teacher_conflict',
 'section_conflict',
 'total_conflicts',
 'is_feasible_numeric',
 'is_feasible',
 'preference_score']

In [91]:
print(candidate_df.columns.tolist())
print("\n")
print(df_candidates_scored.columns.tolist())

['subject_id', 'subject_name', 'record_type', 'teacher_id', 'teacher_code', 'teacher_name', 'max_subjects', 'day', 'day_num', 'slot_id', 'start_time', 'end_time', 'start_minutes', 'end_minutes', 'duration_hours', 'weekly_periods', 'consecutive_periods', 'discipline', 'section']


['source_file', 'academic_year', 'semester', 'discipline', 'section', 'batch', 'day', 'subject_id', 'subject_name', 'teacher_id', 'teacher_code', 'room', 'start_time', 'end_time', 'duration_hours', 'record_type', 'duplicate_record', 'missing_subject', 'missing_teacher', 'invalid_time_range', 'invalid_duration', 'overlaps_with_another_class', 'start_minutes', 'end_minutes', 'day_num', 'is_lab', 'is_theory', 'is_remedial', 'has_batch', 'is_morning', 'class_id', 'teacher_daily_load', 'teacher_weekly_load', 'section_daily_load', 'section_weekly_load', 'is_preferred', 'teacher_conflict', 'section_conflict', 'is_feasible', 'slot_index', 'is_afternoon', 'is_saturday', 'is_week_start', 'teacher_load_ratio', 'section_l

In [102]:
# ============================================================
# ADD TEST DATASET IDENTITY
# ============================================================

df_candidates_scored_new["discipline"] = "CSE"
df_candidates_scored_new["section"] = "CSE1"

print(df_candidates_scored_new[
    [
        "discipline",
        "section",
        "subject_id",
        "subject_name",
        "record_type"
    ]
].head())

  discipline section subject_id    subject_name record_type
0        CSE    CSE1  PCC-CS501  Core Subject 1      theory
1        CSE    CSE1  PCC-CS501  Core Subject 1      theory
2        CSE    CSE1  PCC-CS501  Core Subject 1      theory
3        CSE    CSE1  PCC-CS501  Core Subject 1      theory
4        CSE    CSE1  PCC-CS501  Core Subject 1      theory


In [106]:
# ============================================================
# RECURSIVE TIMETABLE GENERATION
# FOR WEEKLY PERIODS + CONSECUTIVE LABS
# ============================================================

def generate_timetable(candidate_df):

    """
    Generates a complete weekly timetable using ML preference
    scores and recursive backtracking.

    The ML model ranks candidate slots.

    Hard constraints:
        1. Teacher cannot have overlapping classes.
        2. Section cannot have overlapping classes.
        3. Room cannot have overlapping classes.
        4. Teacher cannot exceed max_subjects.
        5. Every subject must receive its required weekly periods.
        6. Consecutive-period requirements must be respected.

    For every scheduling unit:
        Highest preference score is tried first.

    If it causes a conflict later:
        Backtracking removes it and tries the next-best option.
    """

    # ========================================================
    # 1. KEEP ONLY FEASIBLE CANDIDATES
    # ========================================================

    feasible = candidate_df[
        candidate_df["is_feasible"] == 1
    ].copy()
    # Make sure scheduling metadata is available
    for col in ["weekly_periods", "consecutive_periods"]:
        if col not in feasible.columns:
            feasible[col] = candidate_df[col]
    if feasible.empty:

        print("\nNo feasible candidates available.")

        return pd.DataFrame()

    # ========================================================
    # 2. REMOVE DUPLICATE CANDIDATES
    # ========================================================

    feasible = feasible.drop_duplicates(
        subset=[
            "discipline",
            "section",
            "subject_id",
            "teacher_code",
            "day_num",
            "start_minutes",
            "end_minutes",
            "room"
        ]
    )

    # ========================================================
    # 3. BUILD REQUIRED SCHEDULING UNITS
    # ========================================================
    #
    # Example:
    #
    # weekly_periods = 4
    # consecutive_periods = NaN
    #
    # => 4 individual periods
    #
    # weekly_periods = 3
    # consecutive_periods = 2
    #
    # => one 2-period block + one 1-period class
    #
    # weekly_periods = 2
    # consecutive_periods = 2
    #
    # => one 2-period block
    # ========================================================

    scheduling_units = []

    subject_info = (
        feasible[
            [
                "discipline",
                "section",
                "subject_id",
                "weekly_periods",
                "consecutive_periods"
            ]
        ]
        .drop_duplicates(
            subset=[
                "discipline",
                "section",
                "subject_id"
            ]
        )
    )

    for _, subject in subject_info.iterrows():

        weekly_periods = int(
            subject["weekly_periods"]
        )

        consecutive = subject["consecutive_periods"]

        # ----------------------------------------------------
        # No consecutive requirement
        # ----------------------------------------------------

        if pd.isna(consecutive):

            for period_number in range(
                1,
                weekly_periods + 1
            ):

                scheduling_units.append(
                    {
                        "discipline":
                            subject["discipline"],

                        "section":
                            subject["section"],

                        "subject_id":
                            subject["subject_id"],

                        "unit_number":
                            period_number,

                        "block_size":
                            1
                    }
                )

        # ----------------------------------------------------
        # Consecutive requirement
        # ----------------------------------------------------

        else:

            consecutive = int(consecutive)

            remaining = weekly_periods

            unit_number = 1

            while remaining >= consecutive:

                scheduling_units.append(
                    {
                        "discipline":
                            subject["discipline"],

                        "section":
                            subject["section"],

                        "subject_id":
                            subject["subject_id"],

                        "unit_number":
                            unit_number,

                        "block_size":
                            consecutive
                    }
                )

                remaining -= consecutive

                unit_number += 1

            # ------------------------------------------------
            # Remaining single period
            # ------------------------------------------------

            while remaining > 0:

                scheduling_units.append(
                    {
                        "discipline":
                            subject["discipline"],

                        "section":
                            subject["section"],

                        "subject_id":
                            subject["subject_id"],

                        "unit_number":
                            unit_number,

                        "block_size":
                            1
                    }
                )

                remaining -= 1

                unit_number += 1

    # ========================================================
    # PRINT REQUIRED UNITS
    # ========================================================

    print("\n")
    print("=" * 60)
    print("SCHEDULING UNITS")
    print("=" * 60)

    for unit in scheduling_units:

        print(
            f"{unit['subject_id']} "
            f"| Unit {unit['unit_number']} "
            f"| Block size: {unit['block_size']}"
        )

    print(
        "\nTotal scheduling units:",
        len(scheduling_units)
    )

    # ========================================================
    # 4. CREATE CANDIDATE OPTIONS FOR EACH UNIT
    # ========================================================

    unit_candidates = []

    for unit in scheduling_units:

        subject_id = unit["subject_id"]
        block_size = unit["block_size"]

        subject_candidates = feasible[
            feasible["subject_id"] == subject_id
        ].copy()

        options = []

        # ====================================================
        # SINGLE PERIOD
        # ====================================================

        if block_size == 1:

            for _, row in subject_candidates.iterrows():

                options.append(
                    [row]
                )

        # ====================================================
        # CONSECUTIVE BLOCK
        # ====================================================

        else:

            # Group by teacher, day and room.
            #
            # Consecutive periods must belong to the same
            # teacher and room.
            # =================================================

            grouped = subject_candidates.groupby(
                [
                    "teacher_code",
                    "day_num",
                    "room"
                ],
                dropna=False
            )

            for _, group in grouped:

                group = group.sort_values(
                    "start_minutes"
                ).reset_index(drop=True)

                # --------------------------------------------
                # Look for consecutive slots
                # --------------------------------------------

                for i in range(
                    len(group) - block_size + 1
                ):

                    block = group.iloc[
                        i:i + block_size
                    ]

                    valid_block = True

                    # ----------------------------------------
                    # Check adjacent times
                    # ----------------------------------------

                    for k in range(
                        len(block) - 1
                    ):

                        current_end = (
                            block.iloc[k]["end_minutes"]
                        )

                        next_start = (
                            block.iloc[k + 1]["start_minutes"]
                        )

                        if current_end != next_start:

                            valid_block = False
                            break

                    if valid_block:

                        options.append(
                            [
                                block.iloc[k]
                                for k in range(block_size)
                            ]
                        )

        # ====================================================
        # SORT OPTIONS BY TOTAL ML SCORE
        # ====================================================

        options.sort(
            key=lambda option: sum(
                float(row["preference_score"])
                for row in option
            ),
            reverse=True
        )

        unit_candidates.append(
            (
                unit,
                options
            )
        )

    # ========================================================
    # 5. PROCESS MOST RESTRICTED UNITS FIRST
    # ========================================================

    unit_candidates.sort(
        key=lambda x: len(x[1])
    )

    # ========================================================
    # 6. RESOURCE TRACKING
    # ========================================================

    teacher_schedule = set()

    section_schedule = set()

    room_schedule = set()

    # Tracks subjects already assigned to each teacher
    teacher_subjects = {}

    selected = []

    # ========================================================
    # 7. VALIDATE OPTION
    # ========================================================

    def is_valid_option(option):

        if not option:

            return False

        # ----------------------------------------------------
        # Check every row in the block
        # ----------------------------------------------------

        for candidate in option:

            teacher = candidate["teacher_code"]

            section = candidate["section"]

            room = candidate["room"]

            day = candidate["day_num"]

            start = candidate["start_minutes"]

            end = candidate["end_minutes"]

            # ================================================
            # TEACHER CONFLICT
            # ================================================

            if not pd.isna(teacher):

                for (
                    existing_teacher,
                    existing_day,
                    existing_start,
                    existing_end
                ) in teacher_schedule:

                    if (
                        existing_teacher == teacher
                        and
                        existing_day == day
                    ):

                        if (
                            start < existing_end
                            and
                            end > existing_start
                        ):

                            return False

            # ================================================
            # SECTION CONFLICT
            # ================================================

            for (
                existing_section,
                existing_day,
                existing_start,
                existing_end
            ) in section_schedule:

                if (
                    existing_section == section
                    and
                    existing_day == day
                ):

                    if (
                        start < existing_end
                        and
                        end > existing_start
                    ):

                        return False

            # ================================================
            # ROOM CONFLICT
            # ================================================

            if (
                not pd.isna(room)
                and
                str(room).strip() != ""
            ):

                for (
                    existing_room,
                    existing_day,
                    existing_start,
                    existing_end
                ) in room_schedule:

                    if (
                        existing_room == room
                        and
                        existing_day == day
                    ):

                        if (
                            start < existing_end
                            and
                            end > existing_start
                        ):

                            return False

        # ====================================================
        # TEACHER MAX SUBJECT CHECK
        # ====================================================

        for candidate in option:

            teacher = candidate["teacher_code"]

            if pd.isna(teacher):

                continue

            max_subjects = candidate["max_subjects"]

            subject_id = candidate["subject_id"]

            current_subjects = teacher_subjects.get(
                teacher,
                set()
            )

            # -----------------------------------------------
            # Only count a new subject
            # -----------------------------------------------

            if subject_id not in current_subjects:

                if (
                    not pd.isna(max_subjects)
                    and
                    len(current_subjects)
                    >= int(max_subjects)
                ):

                    return False

        return True

    # ========================================================
    # 8. ADD OPTION
    # ========================================================

    def add_option(option):

        for candidate in option:

            teacher = candidate["teacher_code"]

            section = candidate["section"]

            room = candidate["room"]

            day = candidate["day_num"]

            start = candidate["start_minutes"]

            end = candidate["end_minutes"]

            # -----------------------------------------------
            # Teacher
            # -----------------------------------------------

            if not pd.isna(teacher):

                teacher_schedule.add(
                    (
                        teacher,
                        day,
                        start,
                        end
                    )
                )

                if teacher not in teacher_subjects:

                    teacher_subjects[teacher] = set()

                teacher_subjects[teacher].add(
                    candidate["subject_id"]
                )

            # -----------------------------------------------
            # Section
            # -----------------------------------------------

            section_schedule.add(
                (
                    section,
                    day,
                    start,
                    end
                )
            )

            # -----------------------------------------------
            # Room
            # -----------------------------------------------

            if (
                not pd.isna(room)
                and
                str(room).strip() != ""
            ):

                room_schedule.add(
                    (
                        room,
                        day,
                        start,
                        end
                    )
                )

            # -----------------------------------------------
            # Timetable
            # -----------------------------------------------

            selected.append(candidate)

    # ========================================================
    # 9. REMOVE OPTION
    # ========================================================

    def remove_option(option):

        for candidate in option:

            teacher = candidate["teacher_code"]

            section = candidate["section"]

            room = candidate["room"]

            day = candidate["day_num"]

            start = candidate["start_minutes"]

            end = candidate["end_minutes"]

            # -----------------------------------------------
            # Teacher
            # -----------------------------------------------

            if not pd.isna(teacher):

                teacher_schedule.discard(
                    (
                        teacher,
                        day,
                        start,
                        end
                    )
                )

                subject_id = candidate["subject_id"]

                # Remove subject only if no other selected
                # class uses that teacher for the subject.
                # --------------------------------------------

                still_used = any(
                    row["teacher_code"] == teacher
                    and
                    row["subject_id"] == subject_id
                    for row in selected
                    if row is not candidate
                )

                if not still_used:

                    if teacher in teacher_subjects:

                        teacher_subjects[
                            teacher
                        ].discard(
                            subject_id
                        )

            # -----------------------------------------------
            # Section
            # -----------------------------------------------

            section_schedule.discard(
                (
                    section,
                    day,
                    start,
                    end
                )
            )

            # -----------------------------------------------
            # Room
            # -----------------------------------------------

            if (
                not pd.isna(room)
                and
                str(room).strip() != ""
            ):

                room_schedule.discard(
                    (
                        room,
                        day,
                        start,
                        end
                    )
                )

            # -----------------------------------------------
            # Selected
            # -----------------------------------------------

            selected.remove(candidate)

    # ========================================================
    # 10. RECURSIVE BACKTRACKING
    # ========================================================

    recursion_count = [0]

    def backtrack(unit_index):

        recursion_count[0] += 1

        # ====================================================
        # BASE CASE
        # ====================================================

        if unit_index == len(unit_candidates):

            return True

        unit, options = unit_candidates[unit_index]

        # ====================================================
        # TRY OPTIONS IN ML PREFERENCE ORDER
        # ====================================================

        for option in options:

            # -----------------------------------------------
            # Check hard constraints
            # -----------------------------------------------

            if not is_valid_option(option):

                continue

            # -----------------------------------------------
            # Select
            # -----------------------------------------------

            add_option(option)

            # -----------------------------------------------
            # Recursive call
            # -----------------------------------------------

            if backtrack(
                unit_index + 1
            ):

                return True

            # -----------------------------------------------
            # Backtrack
            # -----------------------------------------------

            remove_option(option)

        return False

    # ========================================================
    # 11. START
    # ========================================================

    print("\n")
    print("=" * 60)
    print("STARTING RECURSIVE TIMETABLE GENERATION")
    print("=" * 60)

    success = backtrack(0)

    # ========================================================
    # 12. RESULT
    # ========================================================

    print(
        "\nRecursive attempts:",
        recursion_count[0]
    )

    if success:

        print(
            "\n✓ COMPLETE TIMETABLE GENERATED"
        )

    else:

        print(
            "\n✗ COULD NOT SCHEDULE ALL REQUIRED PERIODS"
        )

    # ========================================================
    # 13. CREATE DATAFRAME
    # ========================================================

    timetable = pd.DataFrame(selected)

    # ========================================================
    # 14. DISPLAY TIMES
    # ========================================================

    if not timetable.empty:

        timetable["start_time_display"] = (
            timetable["start_minutes"]
            .apply(minutes_to_time)
        )

        timetable["end_time_display"] = (
            timetable["end_minutes"]
            .apply(minutes_to_time)
        )

    return timetable

In [104]:
df_candidates_scored_new.columns.tolist()

['subject_id',
 'subject_name',
 'record_type',
 'teacher_id',
 'teacher_code',
 'teacher_name',
 'max_subjects',
 'day',
 'day_num',
 'slot_id',
 'start_time',
 'end_time',
 'start_minutes',
 'end_minutes',
 'duration_hours',
 'weekly_periods',
 'consecutive_periods',
 'is_lab',
 'is_theory',
 'is_remedial',
 'has_batch',
 'is_morning',
 'is_afternoon',
 'is_saturday',
 'is_week_start',
 'teacher_load_ratio',
 'section_load_ratio',
 'teacher_conflict',
 'section_conflict',
 'total_conflicts',
 'is_feasible_numeric',
 'is_feasible',
 'preference_score',
 'discipline',
 'section']